# 02 - Camada Bronze
Ingestão bruta das avaliações. Sem transformação de negócio — texto e metadados
exatamente como vieram na origem, apenas com colunas de auditoria.

In [0]:
from pyspark.sql import functions as F

CATALOG = "voc_project"
BRONZE_SCHEMA = f"{CATALOG}.bronze"
RAW_PATH = "/Volumes/voc_project/bronze/raw_files"

ARQUIVO = "Womens Clothing E-Commerce Reviews.csv"

## Ingestão
O CSV de reviews tem texto livre com quebras de linha e aspas internas, então usamos
as opções de parsing (multiLine, quote, escape) — mesma lição do projeto anterior.

In [0]:
import re

# 1. Lê o CSV primeiro
df = (spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(f"{RAW_PATH}/{ARQUIVO}"))

# 2. Sanitiza os nomes das colunas (troca caracteres inválidos por _)
novos_nomes = [re.sub(r'[ ,;{}()\n\t=]', '_', c) for c in df.columns]
df = df.toDF(*novos_nomes)

# 3. Adiciona metadados
df = (df
    .withColumn("_ingest_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.lit(ARQUIVO)))

# 4. Grava
df.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{BRONZE_SCHEMA}.reviews")

print(f"✅ bronze.reviews: {df.count()} linhas")

In [0]:
df_bronze = spark.table(f"{BRONZE_SCHEMA}.reviews")
print("Colunas:", df_bronze.columns)
display(df_bronze.limit(10))

## Validação
Contagem vs origem + confirmação de que o texto das reviews não foi corrompido no parsing.

In [0]:
linhas_origem = spark.read.option("header", True).option("multiLine", True) \
    .option("quote", '"').option("escape", '"') \
    .csv(f"{RAW_PATH}/{ARQUIVO}").count()

linhas_tabela = spark.table(f"{BRONZE_SCHEMA}.reviews").count()

status = "✅ PASSOU" if linhas_origem == linhas_tabela else "❌ FALHOU"
print(f"{status} — origem: {linhas_origem}, tabela: {linhas_tabela}")

In [0]:
# Confere se a coluna de texto tem conteúdo real e não ficou desalinhada
df_bronze = spark.table(f"{BRONZE_SCHEMA}.reviews")

# quantas reviews têm texto não-nulo
com_texto = df_bronze.filter(
    F.col("Review_Text").isNotNull() & (F.col("Review_Text") != "")
).count()

print(f"Reviews com texto: {com_texto} de {df_bronze.count()}")
print(f"Reviews sem texto: {df_bronze.count() - com_texto}")

# amostra pra conferir visualmente que o texto está íntegro
display(df_bronze.select("Review_Text", "Rating", "Age", "Department_Name").limit(5))